# Self-RAG Inference Generator (Kaggle T4 x2)

Run the standalone Self-RAG answer generator over MS MARCO using `selfrag/selfrag_llama2_7b` with the HuggingFace 4-bit fallback backend. Use Kaggle GPU T4 x2 with Internet enabled.

This notebook avoids installing vLLM by default because recent vLLM wheels can upgrade CUDA/numpy/protobuf packages and conflict with Kaggle's preinstalled environment. It creates new Self-RAG inference outputs only and does not modify or use the Self-RAG verifier experiment.

In [ ]:
# Kaggle bootstrap. Re-run after enabling GPU T4 x2 and Internet.
# Do not install vLLM in Kaggle's shared environment unless you are using a fresh image;
# recent vLLM wheels can upgrade CUDA/numpy/protobuf packages and trigger resolver conflicts.
!pip install -q accelerate bitsandbytes sentence-transformers faiss-cpu rouge-score

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

CONFIG_PATH = "configs/experiments/self_rag_inference.yaml"
print("Project root:", PROJECT_ROOT)
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES", "default"))

In [ ]:
from rag_filtering.config.loader import load_yaml

cfg = load_yaml(CONFIG_PATH)
assert cfg["model"]["backend"] == "hf", "Kaggle notebook expects the HF 4-bit backend."
assert cfg["hf_fallback"]["load_in_4bit"] is True

print("Model:", cfg["model"]["name"])
print("Backend:", cfg["model"]["backend"])
print("4-bit fallback:", cfg["hf_fallback"]["load_in_4bit"])
print("Data:", cfg["data"]["msmarco_csv"])

In [ ]:
# Full 500-row MS MARCO run. This loads the 7B Self-RAG model.
!python experiments/self_rag_inference/run_inference.py --config configs/experiments/self_rag_inference.yaml

In [ ]:
import pandas as pd

results_dir = PROJECT_ROOT / "results" / "self_rag_inference"
predictions_path = results_dir / "msmarco_predictions.csv"
metrics_path = results_dir / "metrics.json"

display(pd.read_csv(predictions_path).head())
print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))

In [ ]:
# Zip outputs for download from Kaggle's Output panel.
!cd results && zip -r self_rag_inference_outputs.zip self_rag_inference